# L16: NLP实战项目


# L16: NLP实战项目

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 使用BERT等预训练模型完成文本分类、命名实体识别等常见NLP任务
2. 构建完整的NLP Pipeline：数据加载→预处理→模型训练→评估→部署
3. 使用HuggingFace Datasets处理大规模文本数据
4. 分析NLP模型的误差并提出改进方案
5. 完成一个完整的NLP项目并撰写技术报告

## 2. 核心知识点

### 2.1 文本分类实战


In [ ]:
"""
L16-text-classification.py
使用BERT进行文本分类的完整流程
"""

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, load_metric
import numpy as np

# 加载数据
dataset = load_dataset("sst2")  # Stanford Sentiment Treebank
print(f"训练集: {len(dataset['train'])} 样本")
print(f"测试集: {len(dataset['validation'])} 样本")

# 加载预训练模型
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 预处理
def preprocess(examples):
    return tokenizer(examples["sentence"], padding=True, truncation=True, max_length=256)

dataset = dataset.map(preprocess, batched=True)

# 评估指标
metric = load_metric("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 训练
training_args = TrainingArguments(
    output_dir="./output",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"  # 关闭W&B等日志
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()
results = trainer.evaluate()
print(f"测试集准确率: {results['eval_accuracy']:.3f}")


### 2.2 命名实体识别（NER）


In [ ]:
"""
L16-ner.py
使用BERT进行命名实体识别
"""

from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# 使用预训练NER模型
model_name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Pipeline方式
ner_pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

text = "Apple Inc. was founded by Steve Jobs in California in 1976."
entities = ner_pipe(text)

print("命名实体识别结果:")
for ent in entities:
    print(f"  {ent['word']:15} -> {ent['entity_group']:5} (分数: {ent['score']:.3f})")

# 批量处理
texts = [
    "Barack Obama was the 44th President of the United States.",
    "Tesla's factory is located in Fremont, California.",
    "The Eiffel Tower is in Paris, France."
]

for text in texts:
    entities = ner_pipe(text)
    print(f"\n文本: {text}")
    print(f"实体: {[e['word'] for e in entities]}")


### 2.3 完整NLP Pipeline


In [ ]:
"""
L16-full-pipeline.py
完整NLP Pipeline：数据加载→预处理→训练→评估→部署
"""

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import pandas as pd
import streamlit as st

class NLPProjectPipeline:
    def __init__(self, task_type):
        self.task_type = task_type
        self.model = None
        self.tokenizer = None
        self.pipeline = None
    
    def load_data(self, path, format='csv'):
        """加载数据"""
        if format == 'csv':
            self.dataset = pd.read_csv(path)
        elif format == 'json':
            self.dataset = pd.read_json(path)
        return self.dataset
    
    def preprocess(self, text_column):
        """预处理"""
        # 这里可以添加自定义预处理逻辑
        self.text_column = text_column
        return self.dataset
    
    def train(self, model_name, label_column):
        """训练模型"""
        if self.task_type == 'classification':
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(
                model_name, num_labels=self.dataset[label_column].nunique()
            )
        # ... 训练逻辑
        print(f"模型训练完成！")
    
    def evaluate(self, test_data):
        """评估模型"""
        if self.pipeline is None:
            self.pipeline = pipeline(
                "text-classification",
                model=self.model,
                tokenizer=self.tokenizer
            )
        
        predictions = self.pipeline(test_data)
        return predictions
    
    def deploy_streamlit(self):
        """部署为Streamlit Web应用"""
        st.title("NLP 模型预测")
        
        text_input = st.text_area("输入文本")
        if st.button("预测"):
            if self.pipeline is None:
                self.pipeline = pipeline("text-classification", model=self.model)
            result = self.pipeline(text_input)
            st.write(result)

# 使用示例
# pipeline = NLPProjectPipeline(task_type='classification')
# pipeline.load_data('sentiment_data.csv')
# pipeline.preprocess('review_text')
# pipeline.train('bert-base-uncased', 'label')
# pipeline.deploy_streamlit()


## 3. 代码示例


In [ ]:
"""
L16-error-analysis.py
NLP模型误差分析与改进
"""

from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
import numpy as np

# 加载模型和数据
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# 测试数据
test_texts = [
    "This movie is absolutely fantastic! I loved every minute of it.",
    "What a waste of time. Complete garbage.",
    "It's okay, nothing special but not bad either.",
    "I'm not sure if I liked it or not.",
    "Best movie I've ever seen in my life!",
    "Terrible acting, boring plot, awful cinematography."
]

# 预测
predictions = classifier(test_texts)

# 分析结果
print("预测结果分析:")
print("=" * 60)
correct_labels = [1, 0, 1, 0, 1, 0]  # 假设的真实标签

for text, pred, true in zip(test_texts, predictions, correct_labels):
    pred_label = 1 if pred['label'] == 'POSITIVE' else 0
    correct = "✓" if pred_label == true else "✗"
    print(f"{correct} 文本: {text[:50]}...")
    print(f"   预测: {pred['label']} ({pred['score']:.3f}), 真实: {'正面' if true else '负面'}")
    print()

# 误差分析
print("\n误差类型分析:")
print("-" * 60)
print("1. 模糊文本：'It's okay' 容易被错误分类")
print("2. 讽刺检测：模型难以识别文本中的讽刺")
print("3. 混合情感：同时包含正面和负面情感的文本")
print("4. 否定处理：'not bad' 等否定短语容易误判")

# 改进建议
print("\n改进方案:")
print("-" * 60)
print("1. 数据增强：添加更多模糊和混合情感的样本")
print("2. 上下文窗口：扩大上下文范围以更好地理解语境")
print("3. 后处理规则：对否定词和模糊词进行规则处理")
print("4. 集成学习：结合多个模型的预测结果")


## 4. 练习题

1. **文本分类项目**: 选择一个公开NLP数据集（如IMDB评论、垃圾邮件检测），完成从数据加载、模型训练、评估到部署的完整流程。
2. **NER项目**: 在CoNLL-2003数据集上训练命名实体识别模型，评估PER、LOC、ORG、MISC四类实体的识别效果。
3. **错误分析**: 选择一个NLP任务，分析模型在哪些类型的样本上容易出错，提出针对性的改进方案。
4. **模型对比**: 在同一数据集上比较至少3种不同的预训练模型（BERT、RoBERTa、ALBERT等）的效果差异。
5. **部署上线**: 将训练好的模型部署为API服务或Web应用，并实现基本的监控和日志功能。

## 5. 延伸阅读

- **数据集**: HuggingFace Datasets — https://huggingface.co/datasets
- **比赛**: Kaggle NLP比赛 — https://www.kaggle.com/competitions?tag=nlp
- **工具**: Streamlit — https://streamlit.io/
- **工具**: Gradio — https://gradio.app/
- **GitHub**: awesome-nlp — https://github.com/keon/awesome-nlp

---

# 附录：课程进度检查清单

## L13 检查点
- [ ] 掌握文本预处理方法
- [ ] 理解词袋模型和TF-IDF原理
- [ ] 能使用预训练词向量进行语义计算

## L14 检查点
- [ ] 能解释注意力机制的原理
- [ ] 理解Transformer架构
- [ ] 能手算注意力矩阵

## L15 检查点
- [ ] 理解预训练-微调范式
- [ ] 掌握BERT和GPT的区别
- [ ] 能使用HuggingFace加载和微调模型

## L16 检查点
- [ ] 能完成完整的NLP项目
- [ ] 掌握模型评估和误差分析方法
- [ ] 了解模型部署的基本方法

---

*文档版本：v1.0 | 适用教材：AI 人工智能基础教程 Week 7-8*
